# Attribution Agent BYOC Example

This notebook demonstrates the BYOC package flow for **Attribution**: request a short-lived CodeArtifact credential from the Bria Engine, install **`bria_attribution`** from the private **`attribution`** repository, then run the local `BriaAttributionAgent` on a sample image and a sample video.

**Prerequisites**

- `BRIA_API_TOKEN` in the environment.
- Python 3.10 or newer.
- Network access to Bria Engine and AWS CodeArtifact.
- CPU is sufficient; pass `device="cuda"` below for faster embedding on an NVIDIA GPU.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads BRIA_API_TOKEN (and friends) from a local .env file, if present

## 1. CodeArtifact Token

Request a short-lived credential for the **`attribution`** PyPI repository. The credential is used as the CodeArtifact password with username `aws`.

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

repository = "attribution"
params = urlencode({"repository": repository})
url = f"https://engine.prod.bria-api.com/v2/auth/access/code_artifact?{params}"
request = Request(
    url,
    headers={
        "api_token": BRIA_API_TOKEN,
        "token": BRIA_API_TOKEN,
        "User-Agent": "BriaPlatform/APIdocs/LLMsAgent",
    },
)

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    error_body = exc.read().decode("utf-8", errors="replace")
    try:
        error_payload = json.dumps(json.loads(error_body), indent=2)
    except json.JSONDecodeError:
        error_payload = error_body or "<empty response body>"
    raise RuntimeError(
        f"CodeArtifact token request failed for repository {repository!r} "
        f"with HTTP {exc.code}: {exc.reason}\n{error_payload}"
    ) from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"CodeArtifact response did not include an authorization token: {payload}")

expiration = result.get("expiration")
print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))

## 2. Install `bria_attribution`

Installs the base package plus the optional `video` extra needed for video attribution.

In [ ]:
import subprocess
import sys
from urllib.parse import quote

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
attribution_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/attribution/simple/"
)


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "bria_attribution[video]",
        "--extra-index-url",
        attribution_index,
    ]
)

## 3. Initialize The Agent

Create a `BriaAttributionAgent` and call `.setup()` once. `device="cpu"` works everywhere; use `device="cuda"` if a GPU is available.

In [ ]:
import logging
import os

from attribution_agent import AttributionModel, BriaAttributionAgent

logging.basicConfig(level=logging.INFO)

agent = BriaAttributionAgent(device="cpu")
agent.setup()
print("BriaAttributionAgent setup complete.")

## 4. Run Image Attribution

`execute_image` accepts a PIL `Image`, computes its embedding, and submits it to the Bria API under the given `model` and `agent` name.

In [ ]:
import io

import requests
from PIL import Image

image_url = "https://bria-test-images.s3.us-east-1.amazonaws.com/bruno/images/desert_alpha_channel.png"
image = Image.open(io.BytesIO(requests.get(image_url).content))

image_request_id = agent.execute_image(
    image=image,
    model=AttributionModel.IMAGE_RMBG_1,
    api_token=BRIA_API_TOKEN,
    agent="bria-byoc-example",
)

print(f"Image attribution submitted, request_id={image_request_id}")

## 5. Run Video Attribution

`execute_video` accepts a path or URL to a video file, computes stills + motion embeddings, and submits them under the given `model` and `agent` name.

In [ ]:
video_url = "https://bria-test-images.s3.us-east-1.amazonaws.com/videos/5_seconds/person_talking_5s.mp4"

video_request_id = agent.execute_video(
    video=video_url,
    model=AttributionModel.VIDEO_RMBG_1,
    api_token=BRIA_API_TOKEN,
    agent="bria-byoc-example",
)

print(f"Video attribution submitted, request_id={video_request_id}")